# HS10 Data Center Classification

This notebook classifies HS10 commodity codes by their relevance to AI data center construction and operations using Claude AI.

In [ ]:
# Set up Anthropic API key
import os

# # In PowerShell
# $env:ANTHROPIC_API_KEY = "your-new-key-here"

# Option 1: Use environment variable (recommended)
if 'ANTHROPIC_API_KEY' not in os.environ:
    # Option 2: Set it here temporarily (DO NOT commit this to git!)
    os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'
    print("⚠️  API key set from notebook. Remove this before committing!")
else:
    print("✅ Using API key from environment variable")

In [ ]:
# Import required modules
import importlib
import pandas as pd
import hs10_llm_classifier_demo
importlib.reload(hs10_llm_classifier_demo)
from hs10_llm_classifier_demo import classify_single_code, classify_batch, resume_batch_classification

## 1️⃣ Test Single Code Classification

Run this to test the classification on a single HS10 code.

In [ ]:
# Test classify_single_code with a sample HS10 code
code = "8542310040"
description = "PROCESSORS (INCLUDING MICROPROCESSORS): GRAPHICS PROCESSING UNITS (GPUS)"

result = classify_single_code(code, description)

print(f"Code: {code}")
print(f"Description: {description}")
print(f"\nRELEVANCE:  {result['relevance']}")
print(f"CONFIDENCE: {result['confidence']}%")
print(f"CATEGORY:   {result['primary_category']}")
print(f"USE:        {result['specific_use']}")
print(f"REASONING:  {result['reasoning']}")

Code: 8542310040
Description: PROCESSORS (INCLUDING MICROPROCESSORS): GRAPHICS PROCESSING UNITS (GPUS)

RELEVANCE:  High
CONFIDENCE: 100%
CATEGORY:   Compute_Hardware
USE:        GPU accelerators for AI training and inference workloads
REASONING:  GPUs are the core computing components in AI data centers, essential for machine learning training and inference operations. They are the primary hardware driving the demand for AI-focused hyperscale facilities.


## 2️⃣ Batch Process All HS10 Codes

Run this to classify all codes. Automatically saves progress every 10 items and resumes if interrupted.

**Note:** This will take ~3 hours for 22,000 codes. Cost estimate: ~$130

In [ ]:
# BATCH PROCESSING: Classify all HS10 codes with automatic checkpointing

results_df = resume_batch_classification(
    all_codes_file='unique_hs10_commodities.csv',
    checkpoint_file='hs10_classification_progress.csv',
    code_column='I_COMMODITY',
    description_column='I_COMMODITY_LDESC',
    output_file='hs10_classification_final.csv',
    delay=0.5
)

print(f"\n✅ Complete! Total classified: {len(results_df):,} codes")
print(f"Results saved to: hs10_classification_final.csv")

Loading all codes from: unique_hs10_commodities.csv
Total codes to classify: 19,424

Found checkpoint file: hs10_classification_progress.csv
Total rows in checkpoint: 19,424 codes
Found 9,699 error rows (will be retried)
Successfully completed: 9,725 codes
Remaining to classify: 9,699 codes

🚀 Starting classification of 9,699 remaining codes...
Progress will be saved to: hs10_classification_progress.csv
[1/9699] 5206420000: Low (95%) - Not_DC_Related
[2/9699] 5206430000: Low (95%) - Not_DC_Related
[3/9699] 5206440000: Low (95%) - Not_DC_Related
[4/9699] 5206450000: Low (95%) - Not_DC_Related
[5/9699] 5207100000: Low (95%) - Not_DC_Related
[6/9699] 5207900000: Low (95%) - Not_DC_Related
[7/9699] 5208112020: Low (95%) - Not_DC_Related
[8/9699] 5208112040: Low (95%) - Not_DC_Related
[9/9699] 5208112090: Low (95%) - Not_DC_Related
[10/9699] 5208114020: Low (95%) - Not_DC_Related
[11/9699] 5208114040: Low (95%) - Not_DC_Related
[12/9699] 5208114060: Low (95%) - Not_DC_Related
[13/9699] 5208

## 3️⃣ Recovery: If Something Goes Wrong

If the process crashes or gets interrupted, use the options below to recover and continue.

### Option A: Simply Re-run Cell 6

The batch processing cell automatically resumes from the checkpoint. Just run cell 6 again - it will:
- Skip already-completed codes
- Retry any error codes
- Continue from where it stopped

### Option B: Restore from Final File (if checkpoint got corrupted)

If your checkpoint file got messed up but you have a final output file, run the cell below to restore:

In [ ]:
# RESTORE: Copy final file back to checkpoint to continue
import shutil
shutil.copy('hs10_classification_final.csv', 'hs10_classification_progress.csv')

# Verify restoration
df_check = pd.read_csv('hs10_classification_progress.csv')
print(f"✅ Checkpoint restored: {len(df_check):,} rows")
print(f"Errors to retry: {(df_check['relevance'] == 'Error').sum():,} rows")
print("\nNow re-run cell 6 to continue from this checkpoint.")

✅ Checkpoint restored: 19,424 rows
Errors to retry: 9,699 rows
